In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Install Dependencies & Setup Environment

In [ ]:
import os
import sys
import glob
import json
import re
import copy
import warnings
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageOps
from shutil import copyfile
from sklearn.impute import KNNImputer
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold

warnings.filterwarnings('ignore')

# Config Path and Constant

In [ ]:
BASE_INPUT = '/kaggle/input/datasets/akhyaruu/appendicitis'
BASE_OUTPUT = '/kaggle/working/Processed'

# Directory Setup
IMAGE_DIR = os.path.join(BASE_INPUT, 'US_Pictures_kaggle_ready')
TRAIN_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Train')
TEST_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Test')
PREPROC_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Preprocessed')
OUTPUT_DIR = os.path.join(BASE_OUTPUT, 'Dictionaries')

CLINICAL_DATA_FILE = os.path.join(BASE_INPUT, 'app_data.xlsx')

# Blacklist Dummy (jika tidak ada)
BLACKLIST_FILE = os.path.join(BASE_INPUT, 'blacklist.csv')
if not os.path.exists(BLACKLIST_FILE):
    BLACKLIST_FILE = '/kaggle/working/empty_blacklist.csv'
    with open(BLACKLIST_FILE, 'w') as f:
        f.write("image_name\n")

# Konfigurasi
PADDING_MODE = 'constant'
TARGET_LABEL = 'diagnosis'

TABULAR_FEATURES = ['Age', 'Sex', 'Height', 'Weight', 'BMI', 'Alvarado_Score', 'Paedriatic_Appendicitis_Score',
                    'Peritonitis', 'Migratory_Pain', 'Lower_Right_Abd_Pain', 'Contralateral_Rebound_Tenderness', 
                    'Coughing_Pain', 'Psoas_Sign', 'Nausea', 'Loss_of_Appetite', 'Body_Temperature', 'Dysuria', 
                    'Stool', 'WBC_Count', 'Neutrophil_Percentage', 'CRP', 'Ketones_in_Urine', 'RBC_in_Urine', 
                    'WBC_in_Urine', 'Appendix_on_US', 'Appendix_Diameter', 'Free_Fluids', 
                    'Appendix_Wall_Layers', 'Target_Sign', 'Perfusion', 'Perforation', 
                    'Surrounding_Tissue_Reaction', 'Pathological_Lymph_Nodes', 'Bowel_Wall_Thickening', 'Ileus', 
                    'Coprostasis', 'Meteorism', 'Enteritis', 'Appendicular_Abscess', 
                    'Conglomerate_of_Bowel_Loops', 'Gynecological_Findings']

REAL_VALUED = ['Age', 'Height', 'Weight', 'BMI', 'Body_Temperature', 'WBC_Count', 'Neutrophil_Percentage', 
               'CRP', 'Appendix_Diameter']

CONCEPTS = ['Appendix_on_US', 'Free_Fluids', 'Appendix_Wall_Layers', 'Target_Sign', 'Surrounding_Tissue_Reaction',
            'Pathological_Lymph_Nodes', 'Bowel_Wall_Thickening', 'Coprostasis', 'Meteorism', 'Enteritis', 
            'Appendix_Diameter', 'Perforation', 'Appendicular_Abscess', 'Conglomerate_of_Bowel_Loops', 
            'Gynecological_Findings']

# Buat direktori output
for d in [TRAIN_IMAGE_DIR, TEST_IMAGE_DIR, PREPROC_IMAGE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Konfigurasi Path Selesai.")